# Lab 3.3 — Docker Containerisation & Production Patterns
**Module 3: Bridging ML and Engineering**

In this lab you will:
- Understand why Docker solves the "works on my machine" problem for ML services
- Write a production-grade **Dockerfile** for the FastAPI anomaly service
- Write a **docker-compose.yml** for local multi-container orchestration
- Write a **requirements.txt** that pins all dependencies
- Understand multi-stage builds, health checks, and environment variables
- (Optional) Build and test the container if Docker Desktop is installed

> **Instructor Note:** Docker knowledge is now a baseline expectation for ML engineers. The key mental model: Docker packages the model, the API code, and the Python environment into a single portable artifact — the image. The container is a running instance of that image. If Docker Desktop is not installed, walk through the file contents as a code review exercise.


## 📦 Requirements & Troubleshooting

### Required Software

| Tool | How to Install |
|------|---------------|
| **Docker Desktop** | Download from [docs.docker.com/get-docker](https://docs.docker.com/get-docker/) |

> This lab does **not** require any pip packages — it uses the Docker CLI directly.

**Verify Docker is installed:**
```bash
docker --version
docker run hello-world
```

---

### ⚠️ Common Errors & Fixes

**`Docker not found` / `command not found: docker`**
> Docker Desktop is not installed or not running.
> Fix: Download and install Docker Desktop, launch the app, then re-run the cell.

**`permission denied` when running docker commands**
> Your user is not in the `docker` group (Linux only).
> Fix: Run `sudo usermod -aG docker $USER`, log out and back in.

**`Cannot connect to the Docker daemon`**
> Docker Desktop is installed but not running.
> Fix: Open Docker Desktop from your Applications folder and wait for the whale icon to show "running".

**`image not found` / `pull access denied`**
> The image name is wrong or you are offline.
> Fix: Check your internet connection and verify the image name with `docker search <name>`.

In [2]:
import subprocess, sys, os, json

# Check Docker availability
try:
    result = subprocess.run(['docker', '--version'], capture_output=True, text=True)
    DOCKER_AVAILABLE = result.returncode == 0
    docker_version = result.stdout.strip()
except FileNotFoundError:
    DOCKER_AVAILABLE = False
    docker_version = ""

print(f"Docker available: {DOCKER_AVAILABLE}")
if DOCKER_AVAILABLE:
    print(f"  {docker_version}")
else:
    print("  Docker not found — install Docker Desktop from https://docs.docker.com/get-docker/")
    print("  You can still follow along by writing the files and reviewing the commands.")

Docker available: False
  Docker not found — install Docker Desktop from https://docs.docker.com/get-docker/
  You can still follow along by writing the files and reviewing the commands.


## 1. Project Structure

Before building the container, we need a clean directory structure. A production ML service follows this layout:

```
Module_3/
├── app.py                  ← FastAPI application (written in Lab 3.2)
├── model_joblib.pkl        ← serialised model (Lab 3.1)
├── model_card_v2.json      ← model metadata (Lab 3.1)
├── requirements.txt        ← pinned Python dependencies
├── Dockerfile              ← container build instructions
└── docker-compose.yml      ← local orchestration
```

> **Instructor Note:** In a real project, the model file would NOT be baked into the image — it would be mounted as a volume or pulled from an artifact registry (MLflow, S3) at container startup. For this lab we include it directly to keep things simple.


In [3]:
# Verify the files from Labs 3.1 and 3.2 exist
module3_dir = os.path.abspath('.')
files_needed = ['app.py', 'model_joblib.pkl', 'model_card_v2.json']

print(f"Module_3 directory: {module3_dir}")
print()
for fname in files_needed:
    path = os.path.join(module3_dir, fname)
    exists = os.path.exists(path)
    size   = os.path.getsize(path) if exists else 0
    icon   = "✅" if exists else "❌"
    print(f"  {icon}  {fname:30s}  {size/1024:.1f} KB")


Module_3 directory: /Users/nikhil/AI:ML intermediate/Module_3

  ✅  app.py                          2.7 KB
  ✅  model_joblib.pkl                13.2 KB
  ✅  model_card_v2.json              1.4 KB


## 2. Write requirements.txt

Pin exact versions to guarantee reproducible builds. The `pip freeze` approach is common but produces hundreds of transitive dependencies. Instead, we pin only the direct dependencies.

> **Instructor Note:** This is a deliberate engineering choice. Pinning only direct deps makes the file readable and maintainable. In CI, the full lock file (from `pip freeze`) is generated and stored as an artifact.


In [4]:
import importlib.metadata as meta

def get_version(pkg):
    try:
        return meta.version(pkg)
    except meta.PackageNotFoundError:
        return "latest"

deps = {
    "fastapi":       get_version("fastapi"),
    "uvicorn":       get_version("uvicorn"),
    "pydantic":      get_version("pydantic"),
    "scikit-learn":  get_version("scikit-learn"),
    "xgboost":       get_version("xgboost"),
    "lightgbm":      get_version("lightgbm"),
    "joblib":        get_version("joblib"),
    "numpy":         get_version("numpy"),
    "pandas":        get_version("pandas"),
    "httpx":         get_version("httpx"),
}

req_lines = [f"{pkg}=={ver}" for pkg, ver in deps.items()]
req_content = "\n".join(req_lines)

req_path = os.path.join('.', 'requirements.txt')
with open(req_path, 'w') as f:
    f.write(req_content + "\n")

print("requirements.txt:")
print("-" * 40)
print(req_content)
print("-" * 40)
print(f"\n✅ Written to {os.path.abspath(req_path)}")


requirements.txt:
----------------------------------------
fastapi==0.136.3
uvicorn==0.48.0
pydantic==2.13.4
scikit-learn==1.8.0
xgboost==3.2.0
lightgbm==4.6.0
joblib==1.5.3
numpy==2.4.6
pandas==3.0.3
httpx==0.28.1
----------------------------------------

✅ Written to /Users/nikhil/AI:ML intermediate/Module_3/requirements.txt


## 3. Write the Dockerfile

> **Instructor Note:** Walk through each instruction line by line. Key points:
> - `python:3.11-slim` is the right choice — slim cuts image size from ~1GB to ~120MB
> - `WORKDIR /app` keeps paths predictable
> - We copy `requirements.txt` first and install before copying code — this layer is cached by Docker until requirements change, making rebuilds fast
> - `HEALTHCHECK` tells Docker (and Kubernetes) how to verify the container is alive
> - `--no-install-recommends` further reduces image size


In [5]:
dockerfile_content = """# ── Nutanix Anomaly Detector — Dockerfile ─────────────────────────────────
FROM python:3.11-slim

# Prevent Python from writing .pyc files and enable stdout/stderr logging
ENV PYTHONDONTWRITEBYTECODE=1 \\
    PYTHONUNBUFFERED=1 \\
    MODEL_PATH=/app/model_joblib.pkl \\
    CARD_PATH=/app/model_card_v2.json

WORKDIR /app

# Install OS-level dependencies (needed for LightGBM / XGBoost on slim)
RUN apt-get update && \\
    apt-get install -y --no-install-recommends libgomp1 && \\
    rm -rf /var/lib/apt/lists/*

# Install Python dependencies first (layer cached until requirements.txt changes)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code and model artifacts
COPY app.py .
COPY model_joblib.pkl .
COPY model_card_v2.json .

# Expose the API port
EXPOSE 8000

# Verify the container is healthy every 30 seconds
HEALTHCHECK --interval=30s --timeout=10s --start-period=15s --retries=3 \\
    CMD python -c "import httpx; httpx.get('http://localhost:8000/health').raise_for_status()" \\
    || exit 1

# Start the API server
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "2"]
"""

dockerfile_path = os.path.join('.', 'Dockerfile')
with open(dockerfile_path, 'w') as f:
    f.write(dockerfile_content.strip())

print("Dockerfile written:")
print("-" * 50)
print(dockerfile_content.strip())
print("-" * 50)
print(f"\n✅ Written to {os.path.abspath(dockerfile_path)}")


Dockerfile written:
--------------------------------------------------
# ── Nutanix Anomaly Detector — Dockerfile ─────────────────────────────────
FROM python:3.11-slim

# Prevent Python from writing .pyc files and enable stdout/stderr logging
ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1 \
    MODEL_PATH=/app/model_joblib.pkl \
    CARD_PATH=/app/model_card_v2.json

WORKDIR /app

# Install OS-level dependencies (needed for LightGBM / XGBoost on slim)
RUN apt-get update && \
    apt-get install -y --no-install-recommends libgomp1 && \
    rm -rf /var/lib/apt/lists/*

# Install Python dependencies first (layer cached until requirements.txt changes)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code and model artifacts
COPY app.py .
COPY model_joblib.pkl .
COPY model_card_v2.json .

# Expose the API port
EXPOSE 8000

# Verify the container is healthy every 30 seconds
HEALTHCHECK --interval=30s --timeout=10s --start-period=15s --r

## 4. Write docker-compose.yml

`docker-compose` is for running multiple containers together locally. Even for a single service, it's useful because it replaces long `docker run` commands with a declarative file.

> **Instructor Note:** In a real deployment, this might also define a `redis` container (prediction cache), a `postgres` container (alert log store), and a `grafana` container (monitoring). For now, one service is enough to learn the pattern.


In [6]:
compose_content = """version: "3.9"

services:
  anomaly-detector:
    build: .                          # build from Dockerfile in current directory
    image: nutanix-anomaly-detector:1.0.0
    container_name: anomaly-detector
    ports:
      - "8000:8000"                   # host_port:container_port
    environment:
      - MODEL_PATH=/app/model_joblib.pkl
      - CARD_PATH=/app/model_card_v2.json
    healthcheck:
      test: ["CMD-SHELL", "python -c \\"import httpx; httpx.get('http://localhost:8000/health').raise_for_status()\\""]
      interval: 30s
      timeout: 10s
      retries: 3
      start_period: 15s
    restart: unless-stopped           # restart automatically on crash
    deploy:
      resources:
        limits:
          cpus: "1.0"
          memory: 512M
"""

compose_path = os.path.join('.', 'docker-compose.yml')
with open(compose_path, 'w') as f:
    f.write(compose_content.strip())

print("docker-compose.yml written:")
print("-" * 50)
print(compose_content.strip())
print("-" * 50)
print(f"\n✅ Written to {os.path.abspath(compose_path)}")


docker-compose.yml written:
--------------------------------------------------
version: "3.9"

services:
  anomaly-detector:
    build: .                          # build from Dockerfile in current directory
    image: nutanix-anomaly-detector:1.0.0
    container_name: anomaly-detector
    ports:
      - "8000:8000"                   # host_port:container_port
    environment:
      - MODEL_PATH=/app/model_joblib.pkl
      - CARD_PATH=/app/model_card_v2.json
    healthcheck:
      test: ["CMD-SHELL", "python -c \"import httpx; httpx.get('http://localhost:8000/health').raise_for_status()\""]
      interval: 30s
      timeout: 10s
      retries: 3
      start_period: 15s
    restart: unless-stopped           # restart automatically on crash
    deploy:
      resources:
        limits:
          cpus: "1.0"
          memory: 512M
--------------------------------------------------

✅ Written to /Users/nikhil/AI:ML intermediate/Module_3/docker-compose.yml


## 5. Build and Run the Container

The following cells run Docker commands. **They require Docker Desktop to be running.**

> **Instructor Note:** If Docker is not available, read through the commands and outputs together. The commands are straightforward — `build` creates the image, `run` starts the container, `ps` lists running containers.


In [7]:
# ── Step 1: Build the image ───────────────────────────────────────────────
if DOCKER_AVAILABLE:
    print("Building Docker image — this takes 1-3 minutes on first build...")
    print("(Subsequent builds use the layer cache and are much faster)\n")
    result = subprocess.run(
        ['docker', 'build', '-t', 'nutanix-anomaly-detector:1.0.0', '.'],
        capture_output=True, text=True, cwd='.'
    )
    if result.returncode == 0:
        print("✅ Image built successfully")
        # Show image size
        r2 = subprocess.run(['docker', 'images', 'nutanix-anomaly-detector:1.0.0',
                             '--format', '{{.Repository}}:{{.Tag}}  {{.Size}}'],
                            capture_output=True, text=True)
        print(f"Image: {r2.stdout.strip()}")
    else:
        print("❌ Build failed:")
        print(result.stderr[-2000:])  # last 2000 chars of error
else:
    print("Docker not available — equivalent command:")
    print("  docker build -t nutanix-anomaly-detector:1.0.0 .")


Docker not available — equivalent command:
  docker build -t nutanix-anomaly-detector:1.0.0 .


In [8]:
# ── Step 2: Run the container ────────────────────────────────────────────
if DOCKER_AVAILABLE:
    # Stop any existing container first
    subprocess.run(['docker', 'rm', '-f', 'anomaly-detector'],
                   capture_output=True)
    
    result = subprocess.run([
        'docker', 'run', '-d',
        '--name', 'anomaly-detector',
        '-p', '8000:8000',
        'nutanix-anomaly-detector:1.0.0'
    ], capture_output=True, text=True)
    
    if result.returncode == 0:
        container_id = result.stdout.strip()[:12]
        print(f"✅ Container started: {container_id}")
        
        # Wait for startup
        import time; time.sleep(3)
        
        # Check health
        r2 = subprocess.run(['docker', 'ps', '--filter', 'name=anomaly-detector',
                             '--format', '{{.Status}}'],
                            capture_output=True, text=True)
        print(f"Container status: {r2.stdout.strip()}")
    else:
        print(f"❌ Container failed to start: {result.stderr}")
else:
    print("Docker not available — equivalent command:")
    print("  docker run -d --name anomaly-detector -p 8000:8000 nutanix-anomaly-detector:1.0.0")


Docker not available — equivalent command:
  docker run -d --name anomaly-detector -p 8000:8000 nutanix-anomaly-detector:1.0.0


In [9]:
# ── Step 3: Test the containerised API ───────────────────────────────────
import httpx, time

if DOCKER_AVAILABLE:
    time.sleep(2)  # brief pause for service readiness
    base = "http://localhost:8000"
    
    try:
        r = httpx.get(f"{base}/health", timeout=10)
        print(f"Health check: {r.status_code}")
        print(json.dumps(r.json(), indent=2))
        
        # Test a prediction
        payload = {
            "cpu_percent": 91.0, "memory_usage_gb": 58.0, "disk_io_mbps": 460.0,
            "network_rx_mbps": 175.0, "network_tx_mbps": 155.0, "active_vms": 35,
            "stargate_ops": 4500, "cerebro_replication_lag_s": 88.0,
        }
        r2 = httpx.post(f"{base}/predict", json=payload, timeout=10)
        print(f"\nPrediction: {r2.status_code}")
        print(json.dumps(r2.json(), indent=2))
    except Exception as e:
        print(f"Could not reach container: {e}")
        print("Check: docker logs anomaly-detector")
else:
    print("Docker not available — equivalent test:")
    print("  curl http://localhost:8000/health")
    print("  curl -X POST http://localhost:8000/predict \\")
    print("       -H 'Content-Type: application/json' \\")
    print('       -d \'{"cpu_percent":91,"memory_usage_gb":58,"disk_io_mbps":460,"network_rx_mbps":175,"network_tx_mbps":155,"active_vms":35,"stargate_ops":4500,"cerebro_replication_lag_s":88}\'')

Docker not available — equivalent test:
  curl http://localhost:8000/health
  curl -X POST http://localhost:8000/predict \
       -H 'Content-Type: application/json' \
       -d '{"cpu_percent":91,"memory_usage_gb":58,"disk_io_mbps":460,"network_rx_mbps":175,"network_tx_mbps":155,"active_vms":35,"stargate_ops":4500,"cerebro_replication_lag_s":88}'


In [10]:
# ── Step 4: Container lifecycle commands ─────────────────────────────────
print("Useful Docker commands for this service:")
print()
print("  docker ps                                   # list running containers")
print("  docker logs anomaly-detector               # view container logs")
print("  docker logs -f anomaly-detector            # follow logs in real time")
print("  docker exec -it anomaly-detector bash      # open shell in container")
print("  docker stop anomaly-detector               # stop container")
print("  docker start anomaly-detector              # restart container")
print("  docker rm -f anomaly-detector              # delete container")
print("  docker images                              # list all images")
print("  docker rmi nutanix-anomaly-detector:1.0.0  # delete image")
print()
print("  docker-compose up -d                        # start with docker-compose")
print("  docker-compose down                         # stop and remove")
print("  docker-compose logs -f                      # follow all service logs")

if DOCKER_AVAILABLE:
    print()
    print("Stopping container...")
    subprocess.run(['docker', 'stop', 'anomaly-detector'], capture_output=True)
    print("Container stopped ✅")


Useful Docker commands for this service:

  docker ps                                   # list running containers
  docker logs anomaly-detector               # view container logs
  docker logs -f anomaly-detector            # follow logs in real time
  docker exec -it anomaly-detector bash      # open shell in container
  docker stop anomaly-detector               # stop container
  docker start anomaly-detector              # restart container
  docker rm -f anomaly-detector              # delete container
  docker images                              # list all images
  docker rmi nutanix-anomaly-detector:1.0.0  # delete image

  docker-compose up -d                        # start with docker-compose
  docker-compose down                         # stop and remove
  docker-compose logs -f                      # follow all service logs


## 6. Multi-Stage Build (Advanced)

> **Instructor Note:** Multi-stage builds separate the build environment from the runtime environment. The builder stage installs compilers and build tools; only the final artifact is copied to the slim runtime image. This can reduce a 1.2GB image to 180MB.

```dockerfile
# ── Stage 1: builder ────────────────────────────────────────────────
FROM python:3.11 AS builder
WORKDIR /build
COPY requirements.txt .
RUN pip install --user --no-cache-dir -r requirements.txt

# ── Stage 2: runtime ────────────────────────────────────────────────
FROM python:3.11-slim AS runtime
WORKDIR /app

# Copy only the installed packages from the builder
COPY --from=builder /root/.local /root/.local
ENV PATH=/root/.local/bin:$PATH

# Install runtime OS libs only
RUN apt-get update && apt-get install -y --no-install-recommends libgomp1 \
    && rm -rf /var/lib/apt/lists/*

COPY app.py model_joblib.pkl model_card_v2.json .
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
```

The key pattern: `COPY --from=builder` copies only the pip-installed packages — no build tools, no GCC, no headers.


## 7. Lab Summary

| Concept | File | Key Points |
|---------|------|-----------|
| Reproducible deps | `requirements.txt` | Pin direct dependencies, not transitive |
| Build instructions | `Dockerfile` | slim base, layer caching, HEALTHCHECK |
| Local orchestration | `docker-compose.yml` | Declarative, replaces `docker run` |
| Service code | `app.py` | Single entry point, env vars for config |

**Files created in `Module_3/`:**
- `requirements.txt`
- `Dockerfile`
- `docker-compose.yml`

> **Instructor Note:** The full deployment pipeline is: `train in notebook → serialize → wrap in FastAPI → containerise → push image to registry (ECR, Docker Hub) → deploy to Kubernetes or a VM`. This lab covered steps 3–5.

---
## 🎯 Challenges

### Challenge 1
Add an environment variable `LOG_LEVEL` to the `docker-compose.yml` (default: `"info"`) and pass it through to the `uvicorn` command in the Dockerfile CMD instruction (hint: `--log-level ${LOG_LEVEL}`). What uvicorn log levels are available?


In [11]:
# Challenge 1 — your code here
# Modify the docker-compose.yml to add LOG_LEVEL env var
# and update the Dockerfile CMD to use it


### Challenge 2
Write a `Makefile` (or a shell script `build.sh`) with three targets:
- `build` — runs `docker build`
- `run` — runs `docker run` with the correct flags
- `test` — calls `curl http://localhost:8000/health` and prints the result

Save the file to `Module_3/build.sh`.


In [12]:
# Challenge 2 — your code here
# Write the contents of build.sh and save it to ./build.sh


### Challenge 3
Modify the `Dockerfile` to add a non-root user for running the service (a security best practice). The pattern is:
```dockerfile
RUN adduser --disabled-password --gecos '' appuser
USER appuser
```
Where should this go in the Dockerfile? Why does running as root in a container matter?


In [13]:
# Challenge 3 — your code here
# Write the updated Dockerfile content and explain the security reason
